# ANPR plate-detection: CPU vs GPU inference benchmark

Before running: **Runtime -> Change runtime type -> T4 GPU**.

Setup on your own computer first:
1. Open drive.google.com
2. Create a folder named `anpr_benchmark`
3. Upload into it:
   - `number_plate_yolov8s_v2.pt` (from `backend/models/`)
   - a sample video, e.g. `vid1.mp4`
   - `yolov8n.pt` (from `backend/models/`) — only needed for the "Double detection" section further down

In [ ]:
!pip install -q ultralytics opencv-python-headless

## Mount Google Drive
This will pop up a Google sign-in / permission prompt — approve it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this if your Drive folder has a different name/location.
DRIVE_FOLDER = '/content/drive/MyDrive/anpr_benchmark'

MODEL_PATH = f'{DRIVE_FOLDER}/number_plate_yolov8s_v2.pt'
VIDEO_PATH = f'{DRIVE_FOLDER}/vid1.mp4'

import os
assert os.path.exists(MODEL_PATH), f"Model not found at {MODEL_PATH} — check the folder/filename in Drive"
assert os.path.exists(VIDEO_PATH), f"Video not found at {VIDEO_PATH} — check the folder/filename in Drive"
print("Found both files in Drive.")

### Alternative: upload directly instead of Drive
Skip this if the Drive cell above worked. Only use this if you'd rather upload through the browser each session.
```python
from google.colab import files
print("Upload the model weight (.pt) first...")
uploaded_model = files.upload()
print("Now upload a sample video (.mp4)...")
uploaded_video = files.upload()
MODEL_PATH = list(uploaded_model.keys())[0]
VIDEO_PATH = list(uploaded_video.keys())[0]
```

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Benchmark — every frame of the video, no decimation

This intentionally does **not** use the production `PROCESSING_FPS=5` stride —
it runs inference on every single decoded frame, so the result is a true
per-frame cost, not a sampled one. Same `conf=0.25`, `imgsz=640` as production
(`backend/app/config.py`).

In [ ]:
import cv2
import time
from ultralytics import YOLO

CONF_THRESHOLD = 0.25
IMGSZ = 640

device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL_PATH)
model.to(device)
print(f"Model loaded on: {device}")

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video has {total_frames} frames total — processing all of them.")

# Warm up (first call always pays a one-time cold-start cost, exclude from timing)
ok, warm_frame = cap.read()
if ok:
    model.predict(warm_frame, conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # rewind after warmup

times = []
detections_found = 0
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_id += 1

    t0 = time.perf_counter()
    results = model.predict(frame, conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)
    if device == "cuda":
        torch.cuda.synchronize()  # ensure GPU work finished before stopping the clock
    dt = time.perf_counter() - t0
    times.append(dt)

    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        detections_found += 1

    if frame_id % 100 == 0:
        print(f"  ...{frame_id}/{total_frames} frames done")

cap.release()

avg_t = sum(times) / len(times)
print(f"\n=== Results ({device.upper()}) — ALL {len(times)} frames processed ===")
print(f"Frames with a detection: {detections_found} ({detections_found/len(times):.1%})")
print(f"avg={avg_t:.4f}s  min={min(times):.4f}s  max={max(times):.4f}s")
print(f"Max sustainable fps: {1/avg_t:.1f}")

## Compare against the known CPU baseline
These numbers were measured directly on the project's CPU-only machine (see `ANPR_FLOW_REPORT.md`, Section 4.1), same model + same settings — also full-video, not sampled.

In [ ]:
cpu_avg = 0.38
cpu_min = 0.35
cpu_max = 1.4

print("=== CPU vs GPU comparison ===")
print(f"{'':12}{'avg':>10}{'min':>10}{'max':>10}{'max fps':>12}")
print(f"{'CPU':12}{cpu_avg:>10.3f}{cpu_min:>10.3f}{cpu_max:>10.3f}{1/cpu_avg:>12.1f}")
print(f"{device.upper():12}{avg_t:>10.3f}{min(times):>10.3f}{max(times):>10.3f}{1/avg_t:>12.1f}")
print(f"\nSpeedup: {cpu_avg/avg_t:.1f}x faster on {device.upper()} (avg case)")

## Double detection (vehicle detection -> crop -> plate detection)

The project's *original* architecture (before it was replaced with direct
full-frame plate detection — see `ANPR_FLOW_REPORT.md` Section 1/6) ran two
models per frame:
1. **Vehicle detection** (YOLOv8n, COCO classes: car/motorcycle/bus/truck) on
   the full frame.
2. **Plate detection** (the same `number_plate_yolov8s_v2.pt`) on *each
   vehicle crop*, not the full frame.

This benchmarks that two-stage approach on the same GPU, so it's directly
comparable to the single-stage numbers above — both cost (two model calls
per vehicle instead of one call per frame) and plate coverage (does cropping
to the vehicle first find plates the full-frame approach misses, or the
reverse?).

**Requires one more file in your Drive `anpr_benchmark` folder:**
`yolov8n.pt` (from `backend/models/yolov8n.pt` — this is the stock
Ultralytics COCO-pretrained weight, not a custom one).

In [ ]:
!pip install -q ultralytics opencv-python-headless

import os
import cv2
import time
import torch
from ultralytics import YOLO

# This cell assumes the earlier Drive-mount cell already ran in this session
# (DRIVE_FOLDER, MODEL_PATH, VIDEO_PATH) and defines everything else itself,
# so it works even if run on its own after a fresh Runtime restart.
device = "cuda" if torch.cuda.is_available() else "cpu"
CONF_THRESHOLD = 0.25
IMGSZ = 640

VEHICLE_MODEL_PATH = f'{DRIVE_FOLDER}/yolov8n.pt'
assert os.path.exists(VEHICLE_MODEL_PATH), f"Vehicle model not found at {VEHICLE_MODEL_PATH} — upload yolov8n.pt to Drive first"

VEHICLE_CONF_THRESHOLD = 0.4
VEHICLE_IMGSZ = 640
VEHICLE_CLASS_MAP = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}  # COCO class ids

vehicle_model = YOLO(VEHICLE_MODEL_PATH)
vehicle_model.to(device)
plate_model_double = YOLO(MODEL_PATH)  # separate instance from the single-stage `model` above, for a clean comparison
plate_model_double.to(device)
print(f"Vehicle + plate models loaded on: {device}")

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Warm up both models (excluded from timing)
ok, warm_frame = cap.read()
if ok:
    vehicle_model.predict(warm_frame, conf=VEHICLE_CONF_THRESHOLD, imgsz=VEHICLE_IMGSZ,
                           classes=list(VEHICLE_CLASS_MAP.keys()), verbose=False)
    plate_model_double.predict(warm_frame, conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

double_times = []          # total time per frame: vehicle detect + all plate-crop detects
vehicle_times = []         # vehicle-detection-only time per frame
plate_times = []           # sum of plate-detection time per frame (across all vehicle crops)
frames_with_plate = 0
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_id += 1

    t_frame_start = time.perf_counter()

    t0 = time.perf_counter()
    vehicle_results = vehicle_model.predict(
        frame, conf=VEHICLE_CONF_THRESHOLD, imgsz=VEHICLE_IMGSZ,
        classes=list(VEHICLE_CLASS_MAP.keys()), verbose=False,
    )
    if device == "cuda":
        torch.cuda.synchronize()
    vehicle_dt = time.perf_counter() - t0
    vehicle_times.append(vehicle_dt)

    plate_dt_total = 0.0
    found_plate_this_frame = False
    vehicle_boxes = vehicle_results[0].boxes
    if vehicle_boxes is not None:
        for vbox in vehicle_boxes:
            vx1, vy1, vx2, vy2 = map(int, vbox.xyxy[0].tolist())
            vehicle_crop = frame[max(0, vy1):vy2, max(0, vx1):vx2]
            if vehicle_crop.size == 0:
                continue

            t1 = time.perf_counter()
            plate_results = plate_model_double.predict(
                vehicle_crop, conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False
            )
            if device == "cuda":
                torch.cuda.synchronize()
            plate_dt_total += time.perf_counter() - t1

            plate_boxes = plate_results[0].boxes
            if plate_boxes is not None and len(plate_boxes) > 0:
                found_plate_this_frame = True

    plate_times.append(plate_dt_total)
    double_times.append(time.perf_counter() - t_frame_start)
    if found_plate_this_frame:
        frames_with_plate += 1

    if frame_id % 100 == 0:
        print(f"  ...{frame_id}/{total_frames} frames done")

cap.release()

double_avg = sum(double_times) / len(double_times)
print(f"\n=== Double detection results ({device.upper()}) — ALL {len(double_times)} frames ===")
print(f"Frames with a plate found: {frames_with_plate} ({frames_with_plate/len(double_times):.1%})")
print(f"Total (vehicle+plate) per frame: avg={double_avg:.4f}s  min={min(double_times):.4f}s  max={max(double_times):.4f}s")
print(f"  of which vehicle-detection: avg={sum(vehicle_times)/len(vehicle_times):.4f}s")
print(f"  of which plate-detection (summed over vehicles in frame): avg={sum(plate_times)/len(plate_times):.4f}s")
print(f"Max sustainable fps: {1/double_avg:.1f}")

In [ ]:
print("=== Single-stage (direct full-frame plate detection) vs Double detection (vehicle -> crop -> plate) ===")
print(f"{'':22}{'avg':>10}{'min':>10}{'max':>10}{'max fps':>12}{'coverage':>12}")
print(f"{'Single-stage':22}{avg_t:>10.4f}{min(times):>10.4f}{max(times):>10.4f}{1/avg_t:>12.1f}{detections_found/len(times):>11.1%}")
print(f"{'Double detection':22}{double_avg:>10.4f}{min(double_times):>10.4f}{max(double_times):>10.4f}{1/double_avg:>12.1f}{frames_with_plate/len(double_times):>11.1%}")
print(f"\nDouble detection is {double_avg/avg_t:.1f}x slower per frame than single-stage (extra model call per vehicle).")
print("Coverage difference shows whether cropping to the vehicle first helps or hurts plate recall on this video.")

## Annotated output video — every frame, with per-frame inference time

`cv2.imshow` has no window to draw into on Colab's headless environment — a
literal copy of `backend/live_preview.py`'s display loop would crash here.
Same detection call, boxes, and per-frame timing overlay as that script, but
instead of a live window this writes an annotated `.mp4` (every frame of the
source, no skipping) and plays it back inline below.

This re-runs inference per frame again (separately from the benchmark cell
above) so the output video's overlay reflects each frame's *own* timing.

In [ ]:
OUTPUT_PATH = "/content/annotated_preview.mp4"

BOX_COLOR = (0, 255, 255)    # yellow, BGR
TEXT_COLOR = (0, 255, 0)     # green, BGR
INFO_COLOR = (0, 255, 255)   # yellow, BGR

cap = cv2.VideoCapture(VIDEO_PATH)
src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Written at the source's own fps since every frame is kept (no stride) —
# the output plays back at the same speed/duration as the source video.
writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), src_fps, (width, height))

frame_id = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_id += 1

    t0 = time.perf_counter()
    results = model.predict(frame, conf=CONF_THRESHOLD, imgsz=IMGSZ, verbose=False)
    if device == "cuda":
        torch.cuda.synchronize()
    infer_ms = (time.perf_counter() - t0) * 1000

    boxes = results[0].boxes
    n_plates = 0 if boxes is None else len(boxes)
    if boxes is not None:
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf.item())
            cv2.rectangle(frame, (x1, y1), (x2, y2), BOX_COLOR, 2)
            cv2.putText(frame, f"{conf:.2f}", (x1, max(0, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, TEXT_COLOR, 2)

    cv2.putText(frame, f"Frame {frame_id}/{total_frames}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, INFO_COLOR, 2)
    cv2.putText(frame, f"Inference: {infer_ms:.1f} ms ({device.upper()})", (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, INFO_COLOR, 2)
    cv2.putText(frame, f"Plates: {n_plates}", (20, 105),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, INFO_COLOR, 2)

    writer.write(frame)

    if frame_id % 100 == 0:
        print(f"  ...{frame_id}/{total_frames} frames annotated")

cap.release()
writer.release()
print(f"Wrote all {frame_id} frames to {OUTPUT_PATH}")

In [ ]:
# Play the annotated video inline in the notebook.
# Note: for a long video this file can be large — base64-inlining it here
# can be slow/heavy in the browser. Use the download cell below instead if
# this cell is too slow to render.
import base64
from IPython.display import HTML

video_b64 = base64.b64encode(open(OUTPUT_PATH, "rb").read()).decode()
HTML(f"""
<video width=800 controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
""")

## Download the annotated video
Recommended for a long video instead of the inline-player cell above — this triggers a browser download of the `.mp4` file.

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)